# Local STARE-PODS Demo — no AWS / no RDS

Mirrors `demo_reconstitute_hdf5_from_s3.py` but uses the **local filesystem** for Parquet storage and **SQLite** for metadata. No cloud credentials needed.

**Core workflow**
1. Ingest a **GMI** and an **SSMIS** granule → Parquet partitions on local disk + SQLite metadata (two instruments so the overlap analytics in step 10 have something to compare)
2. Find intersecting data for a bounding box via STARE SIDs + SQLite (level 4)
3. Load intersecting Parquet partitions from disk
4. Reconstitute an HDF5 file (both S1 and S2 scans) from the level-4 Parquet partitions
5. Compare the reconstituted structure with the original granule
6. Verify SQLite metadata

**Temporal features** (temporal-stare-pods issues 01–06)
7. Temporal catalog — every chunk carries `[t_start, t_end]` + podcode
8. Period-filtered intersection — data-level `[t_start, t_end]` overlap
9. VCF temporal roll-up — union range per pod, on the fly
10. Multi-instrument overlap analytics — the slide-8/9 rendezvous views


In [1]:
#import subprocess, sys
#subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
#                       "/Users/tonhai/workspace/Bayesics/StarePandas_par/stare_demo_add_contruct_parallel",
#                       "-q"])

In [2]:
import os
import sqlite3
import h5py
from starepandas.demo_lib import LocalStarePodsDemo

## Configuration

Edit these paths and parameters before running.

In [3]:
# Parquet store + SQLite DB live here
LOCAL_ROOT = "/tmp/stare_pods_local"

# Resolve the sample granule from the in-repo test-data dir so the notebook is
# safe to run anywhere (no dependency on an external sample directory). Override
# with the STAREPODS_SAMPLE_GRANULE env var to point at your own granule.
import starepandas
_REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(starepandas.__file__)))
GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE",
    os.path.join(
        _REPO_ROOT, "tests", "data", "granules",
        "1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5",
    ),
)

# GMI + SSMIS are a co-located pair (both 2025-01-01, concurrent orbits) whose
# ground tracks cross within ~3 min in 42 shared pods, so the overlap analytics
# (step 10) show genuine multi-instrument rendezvous.
SSMIS_GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE_SSMIS",
    os.path.join(
        _REPO_ROOT, "tests", "data", "granules",
        "1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5",
    ),
)

# STARE partition level used for both ingestion and bbox → SIDs lookup.
# Capped at MAX_PARTITION_LEVEL = 4 (~256 cells/granule), the regime
# where each Parquet partition is multi-MB — ideal for S3.
STARE_LEVEL = 4

# Bounding box filter — set to None to reconstitute the full granule,
# or e.g. (115, -30, 120, -25) to restrict to SW Australia / Perth.
BBOX = None   # (lon_min, lat_min, lon_max, lat_max) or None

DATASETS = ["GMI_S1", "GMI_S2"]

OUTPUT_HDF5 = "/tmp/reconsitution/gmi_local_reconstituted.h5"

# Set to True to wipe LOCAL_ROOT before each run.
# IMPORTANT: re-running without cleaning causes duplicate SQLite entries,
# which inflates the reconstituted HDF5 (e.g. 3× the expected scan count).
# Keep True unless you intentionally want to append more granules.
CLEAN_BEFORE_RUN = True

print(f"Granule    : {os.path.basename(GRANULE_FILE)}")
print(f"SSMIS      : {os.path.basename(SSMIS_GRANULE_FILE)}")
print(f"Datasets   : {DATASETS}")
print(f"BBox       : {BBOX}  (None = full granule)")
print(f"STARE level: {STARE_LEVEL}")
print(f"Local root : {LOCAL_ROOT}")
print(f"Clean first: {CLEAN_BEFORE_RUN}")

Granule    : 1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5
SSMIS      : 1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5
Datasets   : ['GMI_S1', 'GMI_S2']
BBox       : None  (None = full granule)
STARE level: 4
Local root : /tmp/stare_pods_local
Clean first: True


## Step 1 — Ingest granule → local Parquet + SQLite

In [4]:
import shutil

if CLEAN_BEFORE_RUN and os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
    print(f"Removed existing data at {LOCAL_ROOT}")
else:
    print(f"Skipping cleanup (CLEAN_BEFORE_RUN={CLEAN_BEFORE_RUN})")

Removed existing data at /tmp/stare_pods_local


In [5]:
%%time
import time
demo = LocalStarePodsDemo(local_root=LOCAL_ROOT)

local_paths = demo.ingest_granules(GRANULE_FILE, instrument='GMI', level=STARE_LEVEL)
print(f"GMI  : written {len(local_paths)} scan path(s).")

ssmis_paths = demo.ingest_granules(SSMIS_GRANULE_FILE, instrument='SSMIS', level=STARE_LEVEL)
print(f"SSMIS: written {len(ssmis_paths)} scan path(s).")


INFO:starepandas.ingest:Found 1 GMI file(s)


INFO:starepandas.ingest:Processing 1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5


INFO:starepandas.ingest:✓ Stored 1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5 (granule=1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B) → /tmp/stare_pods_local


INFO:starepandas.ingest:Ingested 2 Parquet dataset(s)


INFO:starepandas.ingest:Found 1 SSMIS file(s)


INFO:starepandas.ingest:Processing 1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5


GMI  : written 2 scan path(s).


INFO:starepandas.ingest:✓ Stored 1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5 (granule=1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B) → /tmp/stare_pods_local


INFO:starepandas.ingest:Ingested 4 Parquet dataset(s)


SSMIS: written 4 scan path(s).
CPU times: user 18.8 s, sys: 1.08 s, total: 19.9 s
Wall time: 19.9 s


## Step 2 — Find intersecting data via STARE SIDs

In [6]:
if BBOX is not None:
    location_sids = demo.get_sids_for_bbox(*BBOX, level=STARE_LEVEL)
    print(f"Generated {len(location_sids)} SIDs for bbox {BBOX}")
else:
    location_sids = None
    print("No bbox filter — all partitions will be loaded (full granule reconstitution)")

intersecting = demo.find_intersecting_data(location_sids, instruments=['GMI'])
print(f"Found {len(intersecting)} metadata row(s).")
intersecting[['Dataset', 'grouped_id', 'group_path']]

INFO:starepandas.demo_lib:Loaded all 513 partitions for GMI


No bbox filter — all partitions will be loaded (full granule reconstitution)
Found 513 metadata row(s).


,Dataset,grouped_id,group_path
0,GMI_S1,1819454249457680388,/tmp/stare_pods_local/q30/q302/q3022/q30220/q3...
1,GMI_S1,2206763817411543044,/tmp/stare_pods_local/q33/q331/q3311/q33110/q3...
2,GMI_S1,2269814212194729988,/tmp/stare_pods_local/q33/q333/q3330/q33300/q3...
3,GMI_S1,1826209648898736132,/tmp/stare_pods_local/q30/q302/q3022/q30223/q3...
4,GMI_S1,2276569611635785732,/tmp/stare_pods_local/q33/q333/q3330/q33303/q3...
...,...,...,...
508,GMI_S2,2254051613498933252,/tmp/stare_pods_local/q33/q332/q3322/q33221/q3...
509,GMI_S2,1778921852811345924,/tmp/stare_pods_local/q30/q301/q3011/q30112/q3...
510,GMI_S2,1776670052997660676,/tmp/stare_pods_local/q30/q301/q3011/q30111/q3...
511,GMI_S2,1781173652625031172,/tmp/stare_pods_local/q30/q301/q3011/q30113/q3...


## Step 3 — Load intersecting Parquet partitions from disk

In [7]:
if not intersecting.empty:
    data_dict = demo.download_and_analyze(
        intersecting,
        instruments=list(intersecting['Dataset'].unique()),
    )
    for ds_name, sdf in data_dict.items():
        print(f"{ds_name}: {len(sdf)} rows, columns: {list(sdf.columns[:6])} …")
        display(sdf.head(3))
else:
    print("No intersecting partitions found.")
    data_dict = {}

INFO:starepandas.demo_lib:✓ Combined 659243 rows for GMI_S1


INFO:starepandas.demo_lib:✓ Combined 659243 rows for GMI_S2


GMI_S1: 659243 rows, columns: ['lat', 'lon', 'sids', 'timestamp', 'Tc1', 'Tc2'] …


,lat,lon,sids,timestamp,Tc1,Tc2,Tc3,Tc4,Tc5,Tc6,...,incidenceAngleIndex5,incidenceAngleIndex6,incidenceAngleIndex7,incidenceAngleIndex8,incidenceAngleIndex9,SCstatus_SCorientation,SCstatus_SClatitude,SCstatus_SClongitude,SCstatus_SCaltitude,SCstatus_FractionalGranuleNumber
0,-60.418663,-77.007088,1820290086553930635,2025-01-01 11:29:53.002,173.179993,108.080002,207.080002,159.630005,233.660004,243.289993,...,1,1,1,1,1,180,-65.10408,-74.582634,451.223267,61572.00012
1,-60.431023,-77.121124,1820285685235480459,2025-01-01 11:29:53.002,172.660004,106.129997,206.289993,157.559998,234.039993,241.669998,...,1,1,1,1,1,180,-65.10408,-74.582634,451.223267,61572.00012
2,-60.443966,-77.234947,1820286429792121643,2025-01-01 11:29:53.002,173.179993,106.599998,204.559998,155.669998,232.119995,240.059998,...,1,1,1,1,1,180,-65.10408,-74.582634,451.223267,61572.00012


GMI_S2: 659243 rows, columns: ['lat', 'lon', 'sids', 'timestamp', 'Tc1', 'Tc2'] …


,lat,lon,sids,timestamp,Tc1,Tc2,Tc3,Tc4,Quality,incidenceAngle,...,sunLocalTime,incidenceAngleIndex1,incidenceAngleIndex2,incidenceAngleIndex3,incidenceAngleIndex4,SCstatus_SCorientation,SCstatus_SClatitude,SCstatus_SClongitude,SCstatus_SCaltitude,SCstatus_FractionalGranuleNumber
0,-60.957882,-76.767624,1820203234296677259,2025-01-01 11:29:53.002,264.320007,264.010010,250.850006,259.510010,0,49.57,...,6.319123,1,1,1,1,180,-65.10408,-74.581406,451.223328,61572.00012
1,-60.968994,-76.870178,1820203112318506923,2025-01-01 11:29:53.002,264.339996,263.239990,250.050003,260.750000,0,49.57,...,6.312288,1,1,1,1,180,-65.10408,-74.581406,451.223328,61572.00012
2,-60.980633,-76.972519,1820209139251676299,2025-01-01 11:29:53.002,267.589996,265.049988,250.809998,261.109985,0,49.57,...,6.305466,1,1,1,1,180,-65.10408,-74.581406,451.223328,61572.00012


## Step 4 — Reconstitute HDF5 (S1 + S2)

In [8]:
%%time
import time
granule_basename = os.path.splitext(os.path.basename(GRANULE_FILE))[0]

recon_path = demo.reconstitute_hdf5(
    dataset=DATASETS,
    output_hdf5_path=OUTPUT_HDF5,
    bbox=BBOX,
    granule_name=granule_basename,
)
print(f"Written to: {recon_path}")

INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S1' over bbox=None


INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S2' over bbox=None


INFO:starepandas.demo_lib:✓ Reconstituted HDF5 written to /tmp/reconsitution/gmi_local_reconstituted.h5


Written to: /tmp/reconsitution/gmi_local_reconstituted.h5
CPU times: user 2.1 s, sys: 370 ms, total: 2.47 s
Wall time: 1.82 s


## Step 5 — Structure comparison: reconstituted vs original

In [9]:
def dump_structure(path, label):
    """Print HDF5 group/dataset tree with shapes and dtypes."""
    print(f"\n--- {label} ---")
    with h5py.File(path, "r") as f:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  /{name:50s} {str(obj.shape):20s} {obj.dtype}")
            elif isinstance(obj, h5py.Group) and name != "/":
                print(f"  /{name:50s} Group")
        f.visititems(_visit)

dump_structure(recon_path, f"RECONSTITUTED  ({os.path.basename(recon_path)})")
dump_structure(GRANULE_FILE, f"ORIGINAL       ({os.path.basename(GRANULE_FILE)})")


--- RECONSTITUTED  (gmi_local_reconstituted.h5) ---
  /S1                                                 Group
  /S1/Latitude                                        (2983, 221)          float32
  /S1/Longitude                                       (2983, 221)          float32
  /S1/Quality                                         (2983, 221)          int8
  /S1/SCstatus                                        Group
  /S1/SCstatus/FractionalGranuleNumber                (2983,)              float64
  /S1/SCstatus/SCaltitude                             (2983,)              float32
  /S1/SCstatus/SClatitude                             (2983,)              float32
  /S1/SCstatus/SClongitude                            (2983,)              float32
  /S1/SCstatus/SCorientation                          (2983,)              int16
  /S1/ScanTime                                        Group
  /S1/ScanTime/DayOfMonth                             (2983,)              int8
  /S1/ScanTime/DayOfYear    

## Step 6 — SQLite metadata verification

In [10]:
conn = sqlite3.connect(demo.db_path)
rows = conn.execute(
    'SELECT Dataset, COUNT(*) as cnt FROM "PodsMetadata" GROUP BY Dataset ORDER BY Dataset'
).fetchall()
conn.close()

print(f"SQLite DB: {demo.db_path}")
for dataset_name, cnt in rows:
    print(f"  {dataset_name}: {cnt} partition(s)")

SQLite DB: /tmp/stare_pods_local/metadata.db
  GMI_S1: 265 partition(s)
  GMI_S2: 248 partition(s)
  SSMIS_S1: 370 partition(s)
  SSMIS_S2: 369 partition(s)
  SSMIS_S3: 363 partition(s)
  SSMIS_S4: 367 partition(s)


## Step 7 — Temporal catalog: every chunk carries `[t_start, t_end]` + podcode

Each ingested chunk now records its temporal range and quaternary pod code. `load_local_temporal_catalog` returns the thin projection the analytics use (`podcode / Dataset / t_start / t_end`) — never the heavy `MetadataJson`.

In [11]:
from starepandas.io.granules import load_local_temporal_catalog, load_local_vcf
from starepandas.overlap import (
    rendezvous_events, overlap_matrix, overlap_pod_table, pair_drilldown,
)

catalog = load_local_temporal_catalog(demo.db_path)
print(f"Thin catalog: {len(catalog)} chunks across {catalog['Dataset'].nunique()} datasets")
display(catalog.groupby('Dataset').agg(
    chunks=('podcode', 'size'),
    first_start=('t_start', 'min'),
    last_end=('t_end', 'max'),
))
catalog.head(6)

Thin catalog: 1982 chunks across 6 datasets


,chunks,first_start,last_end
Dataset,,,
GMI_S1,265,2025-01-01 11:29:53.002,2025-01-01 13:03:04.224
GMI_S2,248,2025-01-01 11:29:53.002,2025-01-01 13:03:04.224
SSMIS_S1,370,2025-01-01 11:28:14.868,2025-01-01 13:10:04.874
SSMIS_S2,369,2025-01-01 11:28:14.868,2025-01-01 13:10:04.874
SSMIS_S3,363,2025-01-01 11:28:14.868,2025-01-01 13:10:04.874
SSMIS_S4,367,2025-01-01 11:28:14.868,2025-01-01 13:10:04.874


,podcode,Dataset,t_start,t_end
0,q31301,SSMIS_S1,2025-01-01 11:28:14.868,2025-01-01 13:08:35.636
1,q31301,SSMIS_S3,2025-01-01 11:28:14.868,2025-01-01 13:08:35.636
2,q31301,SSMIS_S2,2025-01-01 11:28:14.868,2025-01-01 13:08:37.534
3,q31301,SSMIS_S4,2025-01-01 11:28:14.868,2025-01-01 13:08:37.534
4,q31300,SSMIS_S1,2025-01-01 11:28:14.868,2025-01-01 13:09:06.015
5,q31300,SSMIS_S2,2025-01-01 11:28:14.868,2025-01-01 13:09:06.015


## Step 8 — Period-filtered intersection

`find_intersecting_data(..., period=(start, end))` keeps only chunks whose **data-level** range `[t_start, t_end]` overlaps the period — ANDed with the spatial pod match. A window bracketing the GMI pass returns its chunks; a window days away returns none. (This is distinct from the granule-level `start_date`/`end_date` filename filter.)

In [12]:
import pandas as pd

gmi = catalog[catalog['Dataset'].str.startswith('GMI')]
gmi_start, gmi_end = gmi['t_start'].min(), gmi['t_end'].max()
match_period = (gmi_start - pd.Timedelta(hours=1), gmi_end + pd.Timedelta(hours=1))
miss_period  = (gmi_start - pd.Timedelta(days=10), gmi_start - pd.Timedelta(days=9))

hit  = demo.find_intersecting_data(None, ['GMI'], period=match_period)
miss = demo.find_intersecting_data(None, ['GMI'], period=miss_period)

print(f"GMI pass window   : [{gmi_start}, {gmi_end}]")
print(f"bracketing period -> {len(hit)} chunks")
print(f"9-10 days earlier -> {len(miss)} chunks")

INFO:starepandas.demo_lib:Loaded all 513 partitions for GMI


GMI pass window   : [2025-01-01 11:29:53.002000, 2025-01-01 13:03:04.224000]
bracketing period -> 513 chunks
9-10 days earlier -> 0 chunks


## Step 9 — VCF temporal roll-up

The temporal hierarchy ("Virtual Collection File") is queryable on the fly: `load_local_vcf(db, level)` groups chunks by their level-`level` ancestor pod and returns each pod's union range `[min(t_start), max(t_end)]` plus its child count. Nothing is materialized — a different level just re-groups the same thin load.

In [13]:
vcf = load_local_vcf(demo.db_path, level=1)
print(f"{len(vcf)} level-1 VCF nodes (one per octant subtree)")
vcf

25 level-1 VCF nodes (one per octant subtree)


,podcode,t_start,t_end,n_chunks,n_without_range
0,q01,2025-01-01 11:33:52.836,2025-01-01 11:40:01.184,52,0
1,q02,2025-01-01 11:50:23.955,2025-01-01 11:58:01.542,76,0
2,q03,2025-01-01 11:36:17.138,2025-01-01 11:52:46.359,160,0
3,q12,2025-01-01 12:26:02.360,2025-01-01 12:39:09.856,55,0
4,q20,2025-01-01 12:37:07.982,2025-01-01 12:44:28.829,40,0
5,q21,2025-01-01 12:58:58.431,2025-01-01 13:02:17.795,16,0
6,q22,2025-01-01 12:42:33.008,2025-01-01 12:59:06.026,140,0
7,q23,2025-01-01 12:41:37.980,2025-01-01 12:59:57.291,138,0
8,q30,2025-01-01 11:29:53.002,2025-01-01 13:03:04.224,42,0
9,q31,2025-01-01 11:28:14.868,2025-01-01 13:10:04.874,136,0


## Step 10 — Multi-instrument overlap analytics (slides 8/9)

`rendezvous_events` sweeps the catalog for passes simultaneously present in a pod within Δt; the matrix / pod-table / drill-downs aggregate that one events frame. This GMI+SSMIS pair is **co-located** — both 2025-01-01, concurrent orbits whose ground tracks cross within ~3 min — so the sweep finds genuine rendezvous (a pod where both instruments have a chunk *and* their pass times fall within Δt: a spatial **and** temporal intersection).

In [14]:
dt = pd.Timedelta(minutes=15)
events = rendezvous_events(catalog, dt)
npods = events['podcode'].nunique() if not events.empty else 0
print(f"Rendezvous over the GMI+SSMIS catalog (dt={dt}): "
      f"{len(events)} events across {npods} shared pods")

print('\nInstrument x instrument matrix — pods where A & B rendezvous (slide 8):')
display(overlap_matrix(events))

print('Per-pod n-way combination counts (slide 9), first 10 pods:')
display(overlap_pod_table(events).head(10))

print('GMI-SSMIS pair drill-down (first 8 shared pods + crossing times):')
display(pair_drilldown(events, 'GMI', 'SSMIS').head(8))

Rendezvous over the GMI+SSMIS catalog (dt=0 days 00:15:00): 118 events across 42 shared pods

Instrument x instrument matrix — pods where A & B rendezvous (slide 8):


,GMI,SSMIS
GMI,0,42
SSMIS,42,0


Per-pod n-way combination counts (slide 9), first 10 pods:


n_instruments,2
podcode,
q22200,1
q22201,1
q22202,1
q22203,1
q23000,1
q23001,1
q23002,1
q23003,1
q23030,1


GMI-SSMIS pair drill-down (first 8 shared pods + crossing times):


,podcode,frequency,times
0,q22200,4,"[2025-01-01 12:57:00.713000, 2025-01-01 12:57:..."
1,q22201,2,"[2025-01-01 12:56:51.101000, 2025-01-01 12:56:..."
2,q22202,2,"[2025-01-01 12:55:24.851000, 2025-01-01 12:55:..."
3,q22203,4,"[2025-01-01 12:56:20.840000, 2025-01-01 12:56:..."
4,q23000,4,"[2025-01-01 12:57:00.713000, 2025-01-01 12:57:..."
5,q23001,4,"[2025-01-01 12:54:57.298000, 2025-01-01 12:54:..."
6,q23002,4,"[2025-01-01 12:57:31.093000, 2025-01-01 12:57:..."
7,q23003,4,"[2025-01-01 12:56:53.118000, 2025-01-01 12:56:..."
